## Imports

In [ ]:
import numpy as np
import xarray as xr
import copy
import src.utils
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cmocean
import os
import pathlib
import cartopy.crs as ccrs
import pandas as pd
import scipy.stats
import tqdm
import time
import xesmf as xe

## Set seaborn plotting style
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})
sns.set_palette("colorblind")

## initialize RNG
rng = np.random.default_rng(seed=100)

## Funcs

In [ ]:
## specify lons/lats for E/W boxes
LONS_E = [240, 280]
LONS_W = [150, 190]


## funcs to plot boxes
def plot_wbox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_W, lats=[-5, 5], **kwargs)


def plot_ebox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_E, lats=[-5, 5], **kwargs)


## funcs to average over regions
def get_eq_avg(data, lons):
    """average over equatorial region and longitudes"""
    return data.sel(longitude=slice(*lons), latitude=slice(-5, 5)).mean(
        ["longitude", "latitude"]
    )


def get_e(data):
    return get_eq_avg(data, LONS_E)


def get_w(data):
    return get_eq_avg(data, LONS_W)


def get_dx(data):
    return get_e(data) - get_w(data)


def add_gridlines(axs):
    """func to add gridlines to axs object"""

    for ax in axs[:-1, 0]:
        ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[],
            ylocs=[-20, 0, 20],
        )
    for ax in axs[-1]:
        gl_ = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            # xlocs=[150, -170, -120, -80],
            xlocs=[],
            ylocs=[-20, 0, 20],
        )
        gl_.top_labels = False
    # gl_.left_labels = False

    return


def plot_level(ax, data, level, ls="-", c="white"):
    """plot single level on hovmoller"""
    ax.contour(
        data.longitude,
        data.latitude,
        data,
        levels=[level],
        colors=c,
        linestyles=ls,
        zorder=10,
        transform=ccrs.PlateCarree(),
    )
    return

import cartopy.crs as ccrs


def make_cb_range(dx, nlevs):
    amp = dx * nlevs - dx / 2

    lev1 = np.arange(0, amp + dx / 2, dx) + dx / 2
    lev0 = -lev1[::-1]
    return np.concatenate([lev0, lev1])


def plot_sst2(ax, data, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance",
        levels=make_cb_range(dx, nlev),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp

def plot_sst_sigma(ax, data, lev_min=.3, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.amp",
        levels=np.arange(lev_min, lev_min+dx*(nlev+1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp

## Data loading

In [ ]:
def sel_area(data, lon_range=None, lat_range=None):
    """select area on t grids"""

    ## find in lon range
    if lon_range is None:
        lon_idx = np.array([True for _ in data.nlon])
    else:
        lon_idx = (
            (data.TLONG >= lon_range[0]) & (data.TLONG <= lon_range[1])
        ).any("nlat").values

    if lat_range is None:
        lat_idx = np.array([True for _ in data.nlat])
    else:
        lat_idx = (
            (data.TLAT >= lat_range[0]) & (data.TLAT <= lat_range[1])
        ).any("nlon").values

    return data.isel(nlon=lon_idx, nlat=lat_idx)

def load_member(i, varname):
    """load member id 'i'"""

    ## set filepath
    load_fp = pathlib.Path("/glade/campaign/collections/gdex/data/d651058/CESM-CAM5-LME/ocn/proc/tseries/monthly/SST")
    pattern = f"b.e11.BLMTRC5CN.f19_g16.{i:03d}.pop.h.SST.*.nc"
    
    ## get files
    files = sorted(list(load_fp.glob(pattern)))

    ## load
    data = xr.open_mfdataset(
        files,
        chunks={"time": 120, "nlat": 384, "nlon": 320},
        decode_timedelta=True,
        parallel=True,
        use_cftime=True,
    )

    ## trim
    data = sel_area(data, lat_range=[-25,25])

    ## get area weights
    area_weight = data["TAREA"].isel(time=0).drop_vars("time").compute()
    area_weight = area_weight/area_weight.max()

    ## mask out NaN values
    ocn_mask = ~np.isnan(data[varname].isel(time=0)).compute()
    data["dA"] = area_weight.where(ocn_mask)

    return data[[varname, "dA"]].squeeze(drop=True).compute()

def area_avg(data, varname, lon_range=None, lat_range=None):
    """average over area"""

    ## trim data
    data_ = sel_area(data, lon_range=lon_range, lat_range=lat_range)

    ## get dims to sum over
    integrate = lambda x : (x * data_["dA"]).sum(["nlon", "nlat"])

    return integrate(data_[varname]) / integrate(1.0)

## Load the data

### init. cluster

In [ ]:
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(n_workers=8)
client = Client(cluster)
client

### do loading

In [ ]:
## loop thru ensemble members
data = []
for m in tqdm.tqdm(np.arange(1,14)):
    data.append(load_member(m, "SST"))

data = xr.concat(data, dim=pd.Index(np.arange(1,14), name="member"))

#### save

In [ ]:
SAVE_FP = pathlib.Path("/glade/work/kcarr/lme_data")
data.to_netcdf(SAVE_FP / "sst.nc")

### regridded version

In [ ]:
import xesmf as xe

# Create the output grid dataset
grid = xr.Dataset(
    {
        "lon": (["lon"], np.arange(0, 360, 1)),
        "lat": (["lat"], np.arange(-30, 31, 1)),
    }
)

# Initialize the regridder
regridder = xe.Regridder(
    data.rename({"TLAT":"lat","TLONG":"lon"}), grid, "bilinear", periodic=True)

# Apply to your data
data_regridded = regridder(data.rename({"TLAT":"lat","TLONG":"lon"}))

#### Save data

In [ ]:
data_regridded.to_netcdf(SAVE_FP / "sst_regridded.nc")

In [ ]:
plt.imshow(
    np.flipud(data["SST"].isel(time=0,member=0).values)
)

In [ ]:
plt.imshow(
    np.flipud(data_regridded["SST"].isel(time=0,member=0).values)
)

In [ ]:
T34 = area_avg(
    sel_area(d, lon_range=(190,240), lat_range=(-5,5)),
    varname="SST",
)

In [ ]:
T34.load()

In [ ]:
np.isnan(d["REGION_MASK"].isel(time=0).values).any()

In [ ]:
d["ocn_mask"] = ~np.isnan(d["SST"].isel(time=0)).compute()

In [ ]:
d["SST"].isel(time=0).where(d["ocn_mask"])

In [ ]:
area_weight = d["TAREA"].isel(time=0).drop_vars("time").compute()
d["area_weight"] = area_weight/area_weight.max()

In [ ]:
def sel_area(data, lon_range=None, lat_range=None):
    """select area on t grids"""

    ## find in lon range
    if lon_range is None:
        lon_idx = np.array([True for _ in data.nlon])
    else:
        lon_idx = (
            (data.TLONG >= lon_range[0]) & (data.TLONG <= lon_range[1])
        ).any("nlat").values

    if lat_range is None:
        lat_idx = np.array([True for _ in data.nlat])
    else:
        lat_idx = (
            (data.TLAT >= lat_range[0]) & (data.TLAT <= lat_range[1])
        ).any("nlon").values

    return data.isel(nlon=lon_idx, nlat=lat_idx)
    # return lon_idx, lat_idx

In [ ]:
d_ = d[["SST","area_weight"]].isel(time=0)

In [ ]:
sel_area(
    d_, lat_range=[-30,30]
).TLAT

In [ ]:
(area_weight/area_weight.max()).values.max()

In [ ]:
d["TAREA"].isel(time=0).values - d["TAREA"].isel(time=1000).values